# EyePACS adaptation of P — supervised DR-specific pre-adaptation of the RGB grader (record §69)

**What this trains.** P itself (ImageNet ConvNeXt-Tiny + GAP + LayerNorm + CORN) on the EyePACS training part of the
pinned patient-level split (31,610 images), with P's optimiser, loss form, augmentation and checkpoint machinery.
Differences from P: the data, batch size 16, and class weights from the EyePACS training counts.

**Selection.** The checkpoint is selected by EyePACS validation QWK only (3,512 images). **No APTOS file is read by
this notebook.** After each seed the selected weights are pinned by SHA-256 (`frozen_checkpoint.json`) and never replaced.

**Input.** The float16 frame cache (Stage 2 + the locked 512 frame), built on the laptop from the raw EyePACS images,
uploaded to Drive and verified here against the split manifest and every shard's SHA-256.

**Nothing downstream runs here**: no APTOS training, no E1-EP / E2-EP, no EyePACS test evaluation, no IDRiD.
`RUN_ADAPTATION` is `False` until the run is approved. GPU runtime (T4). Resumable per seed.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_train as at
import eyepacs_adaptation_data as ea, eyepacs_adaptation_train as et
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: the pinned split, class weights, pretrained weights (nothing is trained here) ====
EXP = colab_config.DRIVE.experiments_root
ROWS = ea.read_split(posixpath.join(REPO_DIR, "dataset_splits"), ea.MANIFEST_SHA256)      # refuses any other manifest
SPLIT = ea.split_report(ROWS)
assert SPLIT["images"] == {"train": 31610, "val": 3512, "total": 35122} and SPLIT["patients_in_both_splits"] == 0
assert not {r["image"] for r in ROWS} & set(ea.BLANK_TRAINING_IMAGES)
COUNTS, CLASS_WEIGHTS = et.class_weights(ROWS)                                           # must equal the recorded weights
assert et.PROTOCOL == dict(at.P_PROTOCOL, batch_size=16)
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256
DRIVE_CACHE = posixpath.join(colab_config.DATASET_ROOT, "EyePACS", "adaptation_frames_v1")
LOCAL_CACHE = "/content/eyepacs_adaptation_frames_v1"
ADAPTATION_SEEDS = (42, 123, 2026)        # must be the seeds fixed in the research record before RUN_ADAPTATION is set
RUN_DIRS = {s: et.run_dir_for(EXP, ea.MANIFEST_SHA256, s) for s in ADAPTATION_SEEDS}
print("split", ea.MANIFEST_SHA256[:16], SPLIT["images"], "| training counts", COUNTS)
print("class weights", [round(w, 4) for w in CLASS_WEIGHTS], "| protocol", et.PROTOCOL)
print("runs:", {s: et.seed_state(d) for s, d in RUN_DIRS.items()})

In [ ]:
# ==== [3] FRAME CACHE: Drive -> local disk (every shard checked by SHA-256), then verified against the split ====
RUN_STAGE_CACHE = True
if RUN_STAGE_CACHE:
    print(ea.stage_cache(DRIVE_CACHE, LOCAL_CACHE))
# staging has just hashed every shard; this verifies identity, completeness, image order and grades against the manifest
FRAMES = ea.FrameCache(LOCAL_CACHE, ROWS, check_hashes=not RUN_STAGE_CACHE, log=print)
print(json.dumps(FRAMES.report, indent=1))
assert FRAMES.report["frames"] == {"train": 31610, "val": 3512}
assert FRAMES.report["identity"]["manifest_sha256"] == ea.MANIFEST_SHA256 and FRAMES.report["identity"]["dtype"] == ea.FRAME_DTYPE
assert ea.FRAME_DTYPE == 'float16'

In [ ]:
# ==== [4] MEMORY / SPEED CHECK: ten optimiser steps of a throwaway model at batch 16. Nothing is saved. ====
RUN_SPEED_CHECK = True
if RUN_SPEED_CHECK:
    import time
    tf.keras.backend.clear_session()
    _, _arrays = pl.load_reference(PRETRAINED)
    _model = et.build_compiled_model(ADAPTATION_SEEDS[0], _arrays, CLASS_WEIGHTS)
    _seq = et.make_epoch_sequence(FRAMES, FRAMES.entries("train"), 0, ADAPTATION_SEEDS[0], et.PROTOCOL["batch_size"], augment=True)
    _times = []
    for _b in range(10):
        _x, _y = _seq[_b]
        _t = time.time()
        _loss = _model.train_on_batch(_x, _y)
        _times.append(time.time() - _t)
    _step = float(np.median(_times[2:]))
    _mem = tf.config.experimental.get_memory_info("GPU:0")
    print(f"seconds per step {_step:.2f} | steps per epoch {len(_seq)} | training part of an epoch ~{_step * len(_seq) / 60:.0f} min "
          f"| peak GPU memory {_mem['peak'] / 2**30:.1f} GiB")
    del _model, _arrays, _seq
    tf.keras.backend.clear_session()

In [ ]:
# ==== [5] THE ADAPTATION RUNS -- THIS TRAINS when RUN_ADAPTATION is True. Resumable per seed. ====
RUN_ADAPTATION = False
if RUN_ADAPTATION:
    tf.keras.backend.clear_session()
    _, REFERENCE_ARRAYS = pl.load_reference(PRETRAINED)
    RESULTS = et.run_sequence(EXP, FRAMES, REFERENCE_ARRAYS, CLASS_WEIGHTS, ADAPTATION_SEEDS, repo_dir=REPO_DIR,
                              staging_root="/content/eyepacs_adaptation_staging")
    for _s, _r in RESULTS.items():
        print(f"seed {_s}: EyePACS validation QWK {_r['frozen']['val_qwk']:.4f} (epoch index {_r['frozen']['best_epoch_index']}, "
              f"stop {_r['stop']['stop_reason']} at {_r['stop']['epoch']}) | frozen {_r['frozen']['sha256']}")
else:
    print("adaptation not run (RUN_ADAPTATION = False):", {s: et.seed_state(d) for s, d in RUN_DIRS.items()})